# 사후(post-hoc) 분석: N-CMAPSS CBM+ 정비 시뮬레이션

**위치.** Phase 3 이후의 분석이며, 원고에는 "post-hoc"로 표시합니다. **실행 전에 이 노트북을 GitHub에 커밋하세요.**
새로 학습하는 모델은 없습니다. 이미 저장된 예측만 씁니다.
- dev 교차검증 예측: Phase 1(PI-BDP V0), Phase 2(RF, GB, LSTM)
- test 예측: Phase 3, 사후 재현 CNN

**질문.** 점예측 정확도가 아니라 **정비 결과**(비계획 고장, 다운타임, 버린 수명)로 비교하면 어떤가? 예측분포를 쓰는 규칙이 점예측 규칙보다 유리한가?

**시뮬레이션 규칙 (실행 전 고정)**
- **판단 시점:** 매 비행이 끝날 때마다 판단합니다(Δt = 1 사이클). 정비를 발령하면 리드타임 L = 5 사이클 뒤에 수행합니다.
- **계획정비 성공 조건:** 발령 시점 t에서 t + L < EOL이면 계획정비로 끝나고, 사용 수명은 t + L이 됩니다. 그렇지 않거나 끝까지 발령하지 않으면 EOL에서 비계획 고장이 납니다.
- **다운타임:** 계획정비 D_p = 2, 비계획 고장 D_u = ρ·D_p, 기본 ρ = 10입니다. 민감도로 ρ ∈ {2, 5, 10, 20, 50}을 봅니다.
- **지표:**
  - 1,000 사이클당 다운타임: test 함대 전체의 (총 다운타임 / 총 운용 사이클) × 1,000
  - 비계획 고장률
  - 평균 미사용 수명 비율: (EOL − 사용 수명) / EOL

**정책**

| 묶음 | 정책 |
|---|---|
| 기준 | CM(고장까지 운용) / TBM(서브셋별 dev 수명의 중도절단 와이블로 구한 최적 교체 주기) / Oracle(EOL−L−1에 발령, 이상적 기준) |
| **조정 없음** (τ = L + Δt = 6) | PI-BDP V0 5% 분위수(보정 전) / **PI-BDP V0 5% 분위수(사전 고정 보정 배율 적용)** / 점예측(RF, GB, LSTM, 문헌 방식 CNN): 예측 ≤ τ이면 발령 |
| **임계값 조정** | τ ∈ {0, …, 40}을 **dev 교차검증 예측**에서 다운타임이 최소가 되도록 고르고(ρ마다, 전 서브셋 공통) test에 적용. 대상: PI-BDP V0 5% 분위수, PI-BDP V0 중앙값, RF, GB, LSTM. 문헌 방식 CNN은 dev 교차검증 예측이 없어 제외 |

- **비교 통계:** 서브셋 → 엔진 계층 붓스트랩 20,000회. 각 정책의 다운타임 95% 구간과, 기준 정책(PI-BDP 보정 5% 분위수, 조정 없음)과의 차이 구간을 냅니다.
- **산출물:** `maint_summary.txt`, 표 CSV, 그림 2개(ρ 민감도, 고장률 대 미사용 수명). 드라이브 `posthoc_maint/`에 저장됩니다.

In [ ]:
import os, sys, json, glob, time, hashlib
import numpy as np, pandas as pd
from scipy.optimize import minimize
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
OUT = f'{ROOT}/posthoc_maint'; os.makedirs(OUT, exist_ok=True)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
C90 = json.load(open(f'{ROOT}/phase1/calibration.json'))['c']['90']; EPS = 0.5
L_LEAD, DT, DP, RHO0 = 5, 1, 2.0, 10
TAU0 = L_LEAD + DT
RHOS = [2, 5, 10, 20, 50]; TAU_GRID = np.arange(0, 41); N_BOOT = 20000
K = ['subset', 'unit', 'cycle']
print('보정 배율 c90 =', C90)

In [ ]:
# === 예측 모으기: 정책별 결정 통계 (사이클 단위) ===
def cal_lo(df):
    m = df.q500.values
    return np.maximum(m - C90 * np.maximum(m - df.q050.values, EPS), 0)

def frame(df, stat):
    d = df[K].copy(); d['stat'] = stat; d['eol'] = (df.cycle + df.true).values
    return d.sort_values(K).reset_index(drop=True)

# --- test ---
v0t = pd.concat([pd.read_csv(f'{ROOT}/phase3/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
TEST = {'PI-BDP q05 (raw)': frame(v0t, v0t.q050.values),
        'PI-BDP q05 (calibrated)': frame(v0t, cal_lo(v0t)),
        'PI-BDP median': frame(v0t, v0t.q500.values)}
for n in ['RF', 'GB', 'LSTM']:
    d = pd.read_csv(f'{ROOT}/phase3/test_{n}.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean()
    TEST[n] = frame(d, d.pred.values)
rc = pd.concat([pd.read_csv(p) for p in glob.glob(f'{ROOT}/posthoc_cnn/pred_*_seed*.csv.gz')], ignore_index=True)
rc = rc.groupby(K + ['seed'], as_index=False)[['pred', 'true']].mean().groupby(K, as_index=False)[['pred', 'true']].mean()
assert set(rc.subset) == set(SUBSETS), '문헌 방식 CNN 예측이 9개 서브셋 모두에 있어야 합니다'
TEST['CNN (lit.-style)'] = frame(rc, rc.pred.values)

# --- dev (교차검증 예측, 임계값 조정용) ---
pr = pd.concat([pd.read_csv(f'{ROOT}/phase1/stageB_preds_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
v0d = pr[(pr.n_h == cfg['n_h']) & (pr.shrink == cfg['shrink'])]
v0d = v0d[[k == cfg['k'][s] for s, k in zip(v0d.subset, v0d.k)]]
DEVP = {'PI-BDP q05 (raw)': frame(v0d, v0d.q050.values), 'PI-BDP median': frame(v0d, v0d.q500.values)}
for n in ['RF', 'GB', 'LSTM']:
    d = pd.read_csv(f'{ROOT}/phase2/preds_{n}.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean()
    DEVP[n] = frame(d, d.pred.values)

# 점검: test 엔진·사이클 구성이 정책마다 같은지
ref = TEST['PI-BDP q05 (raw)'][K]
for n, d in TEST.items(): assert d[K].equals(ref), f'{n}: test 시점 불일치'
print('test 엔진', ref.groupby(['subset', 'unit']).ngroups, '대 | dev 엔진', DEVP['RF'].groupby(['subset', 'unit']).ngroups, '대')

In [ ]:
# === 시뮬레이터 ===
def simulate(d, tau):
    # 엔진별: 첫 발령 시점 t(stat <= tau) -> t + L < EOL 이면 계획정비, 아니면 EOL에서 고장
    out = []
    for (s, u), g in d.groupby(['subset', 'unit'], sort=False):
        eol = int(g.eol.iloc[0]); hit = g.cycle.values[g.stat.values <= tau]
        if len(hit) and hit[0] + L_LEAD < eol: out.append((s, u, eol, hit[0] + L_LEAD, 0))
        else: out.append((s, u, eol, eol, 1))
    return pd.DataFrame(out, columns=['subset', 'unit', 'eol', 'used', 'fail'])

def rate(r, rho):
    dt = np.where(r.fail == 1, rho * DP, DP)
    return 1000 * dt.sum() / r.used.sum()

def summ(r, rho):
    return dict(downtime=rate(r, rho), fail_rate=r.fail.mean(), unused=((r.eol - r.used) / r.eol).mean())

units = ref.drop_duplicates(['subset', 'unit'])[['subset', 'unit']].merge(
    TEST['RF'].groupby(['subset', 'unit']).eol.first().reset_index(), on=['subset', 'unit'])
def fixed(used_fn):
    r = units.copy(); r['used'], r['fail'] = zip(*[used_fn(e) for e in r.eol]); return r
CM = fixed(lambda e: (e, 1))
ORACLE = fixed(lambda e: (e - 1, 0))

# TBM: 서브셋별 dev 수명(중도절단 포함)으로 와이블 적합 -> 최적 교체 주기
def weib(Ls, cens):
    def nll(p):
        k, lam = np.exp(p); z = Ls / lam
        return -((np.log(k / lam) + (k - 1) * np.log(z) - z ** k)[~cens].sum() - (z[cens] ** k).sum())
    k, lam = np.exp(minimize(nll, [np.log(3), np.log(Ls.mean())], method='Nelder-Mead').x); return k, lam
def tbm_age(k, lam, rho):
    T = np.arange(5, 151); grid = np.linspace(0, 150, 3001); R = np.exp(-(grid / lam) ** k)
    cumR = np.concatenate([[0], np.cumsum((R[1:] + R[:-1]) / 2 * np.diff(grid))])
    RT = np.exp(-(T / lam) ** k); IT = np.interp(T, grid, cumR)
    return int(T[np.argmin((DP * RT + rho * DP * (1 - RT)) / IT)])
WB = {}
for s in SUBSETS:
    d = np.load(f'{ROOT}/phase0/dev_{s}.npz'); an = list(d['A_var'])
    eol = pd.Series(d['A'][:, an.index('cycle')]).groupby(d['A'][:, an.index('unit')]).max().values.astype(float); d.close()
    WB[s] = weib(eol, eol >= 100)
def TBM(rho):
    r = units.copy(); T = r.subset.map(lambda s: tbm_age(*WB[s], rho))
    r['used'] = np.where(T < r.eol, T, r.eol); r['fail'] = (T >= r.eol).astype(int); return r
print('와이블(형상, 척도):', {s: tuple(round(x, 2) for x in v) for s, v in WB.items()})

In [ ]:
# === 정책 평가: 조정 없음, dev에서 τ 조정, 기준 정책 (ρ별) ===
rows, TUNED = [], {}
for rho in RHOS:
    rows += [dict(rho=rho, group='reference', policy='CM', tau=np.nan, **summ(CM, rho)),
             dict(rho=rho, group='reference', policy='TBM (Weibull optimal age)', tau=np.nan, **summ(TBM(rho), rho)),
             dict(rho=rho, group='reference', policy='Oracle', tau=np.nan, **summ(ORACLE, rho))]
    for n, d in TEST.items():
        if n == 'PI-BDP median': continue
        rows.append(dict(rho=rho, group='untuned (tau=L+dt)', policy=n, tau=TAU0, **summ(simulate(d, TAU0), rho)))
    for n, dd in DEVP.items():
        dev_rates = [rate(simulate(dd, t), rho) for t in TAU_GRID]
        tb = int(TAU_GRID[int(np.argmin(dev_rates))]); TUNED[(n, rho)] = tb
        rows.append(dict(rho=rho, group='tuned on dev', policy=n, tau=tb, **summ(simulate(TEST[n], tb), rho)))
RES = pd.DataFrame(rows); RES.to_csv(f'{OUT}/maint_table.csv', index=False)
print(RES[RES.rho == RHO0].round(3).to_string(index=False))

In [ ]:
# === 계층 붓스트랩 (ρ = 10): 다운타임 95% 구간, 기준 정책과의 차이 ===
def records(group, policy, rho=RHO0):
    if policy == 'CM': return CM
    if policy == 'Oracle': return ORACLE
    if policy.startswith('TBM'): return TBM(rho)
    tau = TAU0 if group.startswith('untuned') else TUNED[(policy, rho)]
    return simulate(TEST[policy], tau)
rng = np.random.default_rng(1)
subs = sorted(units.subset.unique()); per_unit_idx = {s: np.flatnonzero(units.subset.values == s) for s in subs}
B_idx = []
for _ in range(N_BOOT):
    pick = rng.choice(subs, len(subs))
    B_idx.append(np.concatenate([rng.choice(per_unit_idx[s], len(per_unit_idx[s])) for s in pick]))
def boot_rates(r, rho=RHO0):
    r = r.set_index(['subset', 'unit']).loc[list(zip(units.subset, units.unit))]
    dt = np.where(r.fail.values == 1, rho * DP, DP); used = r.used.values
    return np.array([1000 * dt[i].sum() / used[i].sum() for i in B_idx])
R10 = RES[RES.rho == RHO0].copy()
base = boot_rates(records('untuned', 'PI-BDP q05 (calibrated)'))
ci = []
for _, row in R10.iterrows():
    b = boot_rates(records(row.group, row.policy)); dlt = b - base
    ci.append(dict(lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5),
                   diff_vs_PIcal=row.downtime - R10[(R10.policy == 'PI-BDP q05 (calibrated)') & (R10.group.str.startswith('untuned'))].downtime.iloc[0],
                   diff_lo=np.percentile(dlt, 2.5), diff_hi=np.percentile(dlt, 97.5)))
R10 = pd.concat([R10.reset_index(drop=True), pd.DataFrame(ci)], axis=1)
R10.to_csv(f'{OUT}/maint_rho10_ci.csv', index=False)
print(R10.round(3).to_string(index=False))

In [ ]:
# === 서브셋별 (ρ = 10, 조정 없음 정책 + TBM) ===
per = []
for s in SUBSETS:
    m = units.subset == s
    for n in ['PI-BDP q05 (raw)', 'PI-BDP q05 (calibrated)', 'RF', 'GB', 'LSTM', 'CNN (lit.-style)']:
        r = simulate(TEST[n][TEST[n].subset == s], TAU0); per.append(dict(subset=s, policy=n, **summ(r, RHO0)))
    r = TBM(RHO0)[m.values]; per.append(dict(subset=s, policy='TBM', **summ(r, RHO0)))
PER = pd.DataFrame(per); PER.to_csv(f'{OUT}/maint_per_subset.csv', index=False)
print(PER.pivot(index='subset', columns='policy', values='downtime').round(2).to_string())

In [ ]:
# === 그림: (a) ρ 민감도, (b) 고장률 대 미사용 수명 (조정 없음 정책 τ/분위수 변화, 기술적 곡선) ===
import matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.5, 'legend.frameon': False, 'savefig.dpi': 300})
COL = {'PI-BDP q05 (calibrated)': '#2a78d6', 'PI-BDP q05 (raw)': '#86b6ef', 'GB': '#eb6834', 'LSTM': '#1baf7a',
       'CNN (lit.-style)': '#eda100', 'TBM (Weibull optimal age)': '#52514e', 'RF': '#e87ba4'}
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0))
ax = axs[0]
for pol, c in COL.items():
    g = RES[(RES.policy == pol) & (RES.group != 'tuned on dev')]
    ax.plot(g.rho, g.downtime, '-o', color=c, ms=3.5, lw=2.0 if 'calibrated' in pol else 1.2, label=pol.replace(' (Weibull optimal age)', ''))
ax.set_xscale('log'); ax.set_xticks(RHOS); ax.set_xticklabels(RHOS)
ax.set_xlabel('Unplanned / planned downtime ratio ρ'); ax.set_ylabel('Downtime per 1,000 cycles'); ax.set_title('(a) Untuned policies (τ = L + Δt)', loc='left')
ax.legend(fontsize=6.5)
ax = axs[1]
qcols = {'q025': 0.025, 'q050': 0.05, 'q100': 0.10, 'q250': 0.25, 'q500': 0.5}
pts = [simulate(frame(v0t, v0t[c].values), TAU0) for c in qcols]
ax.plot([((r.eol - r.used) / r.eol).mean() * 100 for r in pts], [r.fail.mean() * 100 for r in pts], '-o', color='#2a78d6', ms=3.5, label='PI-BDP, quantile q varied')
for n in ['GB', 'LSTM', 'CNN (lit.-style)']:
    ps = [simulate(TEST[n], t) for t in range(0, 31, 2)]
    ax.plot([((r.eol - r.used) / r.eol).mean() * 100 for r in ps], [r.fail.mean() * 100 for r in ps], '-o', color=COL[n], ms=2.5, lw=1.0, label=f'{n}, threshold τ varied')
ax.set_xlabel('Mean unused life (%)'); ax.set_ylabel('Unplanned failures (%)'); ax.set_title('(b) Trade-off (descriptive)', loc='left'); ax.legend(fontsize=6.5)
fig.tight_layout(); fig.savefig(f'{OUT}/fig9_maintenance.png', bbox_inches='tight'); fig.savefig(f'{OUT}/fig9_maintenance.pdf', bbox_inches='tight'); plt.show()

In [ ]:
# === 요약 저장 + 해시 ===
txt = '\n'.join([f"Post-hoc CBM+ maintenance simulation {time.strftime('%Y-%m-%d %H:%M:%S')} (NOT pre-registered)",
    f'L={L_LEAD}, dt={DT}, Dp={DP}, rho0={RHO0}, tau0={TAU0}, c90={C90:.3f}; tuned tau chosen on dev LOUO-CV predictions (global over subsets)',
    '', 'rho = 10 with hierarchical bootstrap 95% CI (downtime per 1000 cycles) and difference vs PI-BDP q05 calibrated (untuned):',
    R10[['group', 'policy', 'tau', 'downtime', 'lo', 'hi', 'fail_rate', 'unused', 'diff_vs_PIcal', 'diff_lo', 'diff_hi']].round(3).to_string(index=False),
    '', 'All rho:', RES.round(3).to_string(index=False),
    '', 'Per subset (rho=10, untuned + TBM), downtime per 1000 cycles:', PER.pivot(index='subset', columns='policy', values='downtime').round(2).to_string(),
    '', 'Per subset unplanned failure rate:', PER.pivot(index='subset', columns='policy', values='fail_rate').round(2).to_string()])
open(f'{OUT}/maint_summary.txt', 'w', encoding='utf-8').write(txt)
H = {os.path.basename(p): hashlib.sha256(open(p, 'rb').read()).hexdigest() for p in sorted(glob.glob(f'{OUT}/*')) if 'sha256' not in p}
json.dump(H, open(f'{OUT}/sha256_maint.json', 'w'), indent=1)
print(txt)